In [2]:
# Libraries
import cv2
import numpy as np
from tensorflow.keras.models import load_model
import pygame
import time
import os

# Load MRL model
model_path = r"C:\Users\Bushra Shahid\Desktop\IDVS\MRL Model\model\mrl_best_model.keras"
alarm_path = r"C:\Users\Bushra Shahid\Desktop\IDVS\MRL Model\alarm"

mrl_model = load_model(model_path)

# Initialize pygame for alarm
pygame.mixer.init()

# Class labels (same order as training)
CLASSES = ['open_eyes', 'closed_eyes']

print("MRL Model loaded successfully!")
print("Pygame initialized!")

MRL Model loaded successfully!
Pygame initialized!


In [2]:
# Load face and eye cascades
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade  = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye_tree_eyeglasses.xml')

# Drowsiness tracking
eyes_closed_start = None
eyes_open_start   = None  # New - track when eyes opened
ALARM_THRESHOLD   = 2.0
OPEN_THRESHOLD    = 1.0   # Eyes must be open 1 second to stop alarm
alarm_playing     = False
eye_history       = []
HISTORY_SIZE      = 5

def predict_eye_state(eye_img):
    """Predict eye state from cropped eye image"""
    eye_resized    = cv2.resize(eye_img, (64, 64))
    eye_normalized = eye_resized / 255.0
    eye_input      = eye_normalized.reshape(1, 64, 64, 1)
    prediction     = mrl_model.predict(eye_input, verbose=0)
    raw            = prediction[0][0]
    if raw > 0.5:
        return 'closed_eyes', raw
    else:
        return 'open_eyes', 1 - raw

def get_stable_eye_state(eye_state):
    """Smooth prediction using history"""
    eye_history.append(eye_state)
    if len(eye_history) > HISTORY_SIZE:
        eye_history.pop(0)
    return max(set(eye_history), key=eye_history.count)

def play_alarm():
    """Play alarm sound"""
    global alarm_playing
    alarm_files = [f for f in os.listdir(alarm_path)
                   if f.endswith('.mp3') or f.endswith('.wav')]
    if alarm_files and not alarm_playing:
        pygame.mixer.music.load(os.path.join(alarm_path, alarm_files[0]))
        pygame.mixer.music.play(-1)
        alarm_playing = True
        print("ALARM! Driver drowsy!")

def stop_alarm():
    """Stop alarm sound"""
    global alarm_playing
    if alarm_playing:
        pygame.mixer.music.stop()
        alarm_playing = False

print("Functions ready!")

Functions ready!


In [ ]:
# Start camera
cap = cv2.VideoCapture(0)

eyes_closed_start = None
eyes_open_start   = None

print("Camera started! Press Q to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1,
                                          minNeighbors=5, minSize=(100, 100))

    all_eye_states = []

    for (fx, fy, fw, fh) in faces:
        cv2.rectangle(frame, (fx, fy), (fx+fw, fy+fh), (255, 255, 0), 2)

        # Eye region
        roi_gray  = gray[fy:fy+int(fh*0.6), fx:fx+fw]
        roi_color = frame[fy:fy+int(fh*0.6), fx:fx+fw]

        # Detect eyes
        eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1,
                                            minNeighbors=3, minSize=(20, 20))

        for (ex, ey, ew, eh) in eyes[:2]:
            eye_img    = roi_gray[ey:ey+eh, ex:ex+ew]
            eye_state, confidence = predict_eye_state(eye_img)
            all_eye_states.append(eye_state)

            color = (0, 255, 0) if eye_state == 'open_eyes' else (0, 0, 255)
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), color, 2)
            cv2.putText(roi_color, f"{eye_state[:4]} {confidence:.0%}",
                        (ex, ey-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

    # Determine stable eye state
    if len(all_eye_states) > 0:
        closed_count = all_eye_states.count('closed_eyes')
        open_count   = all_eye_states.count('open_eyes')
        if closed_count >= open_count:
            stable_state = 'closed_eyes'
        else:
            stable_state = 'open_eyes'
        stable_state = get_stable_eye_state(stable_state)
    else:
        stable_state = get_stable_eye_state('closed_eyes')

    # Drowsiness logic
    if stable_state == 'closed_eyes' and len(faces) > 0:
        eyes_open_start = None
        if eyes_closed_start is None:
            eyes_closed_start = time.time()

        elapsed = time.time() - eyes_closed_start

        # Only show text if closed more than 1.0s (ignore blinks)
        if elapsed >= 1.0:
            cv2.putText(frame, f"EYES CLOSED: {elapsed:.1f}s",
                        (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

        if elapsed >= ALARM_THRESHOLD:
            play_alarm()
            cv2.putText(frame, "DROWSINESS ALERT!",
                        (10, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    else:
        eyes_closed_start = None

        # Stop alarm only if eyes open for 1 second continuously
        if alarm_playing:
            if eyes_open_start is None:
                eyes_open_start = time.time()
            open_elapsed = time.time() - eyes_open_start
            cv2.putText(frame, f"Opening... {open_elapsed:.1f}s",
                        (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            if open_elapsed >= 1.0:
                stop_alarm()
                eyes_open_start = None
        else:
            eyes_open_start = None
            if len(faces) > 0:
                cv2.putText(frame, "EYES OPEN",
                            (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow('MRL - Drowsiness Detection', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
stop_alarm()
print("Camera stopped!")

Camera started! Press Q to quit.
ALARM! Driver drowsy!
Camera stopped!
